### Testing RAG Applications 📑

In [3]:
import sys
import types

vertexai_chat_module = types.ModuleType("langchain_community.chat_models.vertexai")
vertexai_chat_module.ChatVertexAI = type("ChatVertexAI", (), {})
sys.modules["langchain_community.chat_models.vertexai"] = vertexai_chat_module

vertexai_llm_module = types.ModuleType("langchain_community.llms.vertexai")
vertexai_llm_module.VertexAI = type("VertexAI", (), {})
sys.modules["langchain_community.llms.vertexai"] = vertexai_llm_module

In [4]:
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from typing import List
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document
from langchain_ollama import ChatOllama

/var/folders/90/zqrw_y6x2_74r6yr2q3qcrn00000gn/T/ipykernel_66470/2351560790.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [5]:
llm = ChatOllama(
    base_url="http://localhost:11434",
    model = "qwen2.5:latest",
    temperature=0.5,
    max_tokens = 250
)

In [6]:
# Load data from Web
loader = WebBaseLoader("https://www.descope.com/learn/post/mcp")
data = loader.load()

# Split text into documents
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
splits = text_splitter.split_documents(data)

# Add text to vector db
embedding = OllamaEmbeddings(model="nomic-embed-text:latest")
vectordb = Chroma.from_documents(documents=splits, embedding=embedding)

# Create a retriever
retriever = vectordb.as_retriever()

def format_docs(docs: List[Document]) -> str:
    return "\n\n".join([d.page_content for d in docs])


template = """Answer the question based only on the following context:

    {context}
    
    Give a summary not the full detail

    Question: {question}
    """
prompt = ChatPromptTemplate.from_template(template)


def retrieve_and_format(question):
    docs = retriever.invoke(question)
    return format_docs(docs)

chain = {"context": retrieve_and_format, "question": RunnablePassthrough()} | prompt | llm | StrOutputParser()


In [7]:
response = chain.invoke("What is MCP")

print(response)

MCP, or Model Context Protocol, standardizes how language models (LLMs) interact with external systems by streaming tool definitions and their capabilities to the LLM. This protocol helps in defining model-specific function schemas and implementing handlers for function calls, making the integration process more standardized and efficient.


In [8]:
from langchain_classic.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(llm=llm, retriever=retriever)

In [9]:
def query_with_context(question):
    retrieved_document = retrieve_and_format(question)
    response = qa_chain.run(question)
    return response, retrieved_document

In [10]:
actual, context = query_with_context("What is MCP")

actual, context

/var/folders/90/zqrw_y6x2_74r6yr2q3qcrn00000gn/T/ipykernel_66470/1524904417.py:3: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  response = qa_chain.run(question)


('MCP stands for Model Context Protocol. It standardizes how Language Models, particularly Large Language Models (LLMs), integrate with external systems and tools. This protocol allows for the streaming of tool definitions—including their capabilities, data stores, and possible prompts—to LLMs from an MCP server. By doing so, it provides a standardized way to define model-specific function schemas and implement handlers, which are necessary when using function calls directly with an LLM API without MCP. This makes the interaction between LLMs and external tools more seamless and context-aware.',
 "## Why this matters\n\nnot mutually exclusive with MCP; the new protocol simply standardizes how this API feature works, adding context for the LLM. This is achieved by streaming tool definitions (including their capabilities, data stores, and possible prompts) to LLMs from the MCP server.\xa0Without MCP, when you use a function call directly with an LLM API, you need to:Define model-specific

### Testing RAG Application with RAGAs


In [11]:
test_data = [
    {
        "input": "What is MCP",
        "reference": "The Model Context Protocol (MCP) addresses this challenge by providing a standardized way for LLMs to connect with external data sources and tools—essentially a “universal remote” for AI apps. Released by Anthropic as an open-source protocol, MCP builds on existing function calling by eliminating the need for custom integration between LLMs and other apps."
    },
    {
        "input": "What is Relationship between function calling & Model Context Protocol",
        "reference": "The Model Context Protocol (MCP) builds on top of function calling, a well-established feature that allows large language models (LLMs) to invoke predetermined functions based on user requests. MCP simplifies and standardizes the development process by connecting AI applications to context while leveraging function calling to make API interactions more consistent across different applications and model vendors."
    },
    {
        "input": "What are the core components of MCP, just give the heading",
        "reference":""" 
                    - MCP Client
                    - MCP Servers
                    - Protocol Handshake
                    - Capability Discovery
                """
    }
]

In [12]:
dataset = []

for question in test_data:
    actual, context = query_with_context(question['input'])
    
    dataset.append({
        "user_input": question['input'],
        "retrieved_contexts": [context],
        "response": actual,
        "reference": question['reference']
    })

dataset


[{'user_input': 'What is MCP',
  'retrieved_contexts': ["## Why this matters\n\nnot mutually exclusive with MCP; the new protocol simply standardizes how this API feature works, adding context for the LLM. This is achieved by streaming tool definitions (including their capabilities, data stores, and possible prompts) to LLMs from the MCP server.\xa0Without MCP, when you use a function call directly with an LLM API, you need to:Define model-specific function schemas, which are JSON descriptions of the function, acceptable parameters, and what it returns.Implement handlers (the\n\nWhat Is the Model Context Protocol (MCP) and How It WorksSkip to main contentArrow RightGet your complimentary copy of the Gartner Report: IAM Adapts to Secure and Enable AI Agents. Let's go >Log InUser CircleProductUse CasesDevelopersCustomersResourcesCompanyPricingSign upArrow RightBook a demoArrow Right# Contents\n\n## Summary\nExplains the Model Context Protocol (MCP), how it standardizes LLM integration wi

In [13]:
from ragas.metrics import LLMContextRecall, NoiseSensitivity, Faithfulness, FactualCorrectness, AnswerRelevancy
from ragas.llms import LangchainLLMWrapper
from ragas import (EvaluationDataset, evaluate)


evaluator_llm = LangchainLLMWrapper(llm)

evaluation_dataset = EvaluationDataset.from_list(dataset)

result = evaluate(dataset=evaluation_dataset, 
                  metrics=[LLMContextRecall(),
                           Faithfulness(),
                           AnswerRelevancy(),
                           FactualCorrectness()],
                  llm = evaluator_llm)

/Users/karthikkk/Downloads/aiqaDemoSource-2/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/90/zqrw_y6x2_74r6yr2q3qcrn00000gn/T/ipykernel_66470/3486702305.py:1: DeprecationWarning: Importing LLMContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextRecall
  from ragas.metrics import LLMContextRecall, NoiseSensitivity, Faithfulness, FactualCorrectness, AnswerRelevancy
/var/folders/90/zqrw_y6x2_74r6yr2q3qcrn00000gn/T/ipykernel_66470/3486702305.py:1: DeprecationWarning: Importing NoiseSensitivity from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import NoiseSens

In [14]:
result.to_pandas()

,user_input,retrieved_contexts,response,reference,context_recall,faithfulness,answer_relevancy,factual_correctness(mode=f1)
0,What is MCP,[## Why this matters\n\nnot mutually exclusive...,MCP stands for Model Context Protocol. It stan...,The Model Context Protocol (MCP) addresses thi...,0.5,0.75,0.829753,0.36
1,What is Relationship between function calling ...,[while building on top of function calling—the...,The relationship between function calling and ...,The Model Context Protocol (MCP) builds on top...,1.0,1.00,0.866584,0.83
2,"What are the core components of MCP, just give...",[What Is the Model Context Protocol (MCP) and ...,MCP Architecture and Core Components,\n - MCP Client\n ...,0.5,1.00,0.910370,0.33
